In [1]:
import pandas

train_df = pandas.read_csv("../../data/processed/step3_area.csv", low_memory=False)
train_df["BIRTH_YMD"] = pandas.to_datetime(train_df["BIRTH_YMD"], errors="coerce")
weather_df = pandas.read_csv("../../data/processed/step_4-3_weather.csv")

print(f"train_df: {train_df.shape}")
print(f"weather_df: {weather_df.shape}")
print(f"\ntrain_df 컬럼: {train_df.columns.tolist()}")
print(f"\nweather_df 컬럼: {weather_df.columns.tolist()}")

print(f"BIRTH_YMD 결측: {train_df['BIRTH_YMD'].isnull().sum():,}")
print(f"BIRTH_YMD 범위: {train_df['BIRTH_YMD'].min()} ~ {train_df['BIRTH_YMD'].max()}")
print(f"shape: {train_df.shape}")

train_df: (2408699, 35)
weather_df: (1645237, 9)

train_df 컬럼: ['sido', 'sigungu', 'eupmyeondong', 'stn', 'ABATT_DATE', 'JUDGE_DATE', 'JUDGE_SEX', 'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'WGRADE', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH', 'COST_AMT', 'AGE', 'BIRTH_YMD', 'CATTLE_NO', 'FARM_UNIQUE_NO', 'LAST_GRADE', 'death_count', 'KPN_NO', 'FATHER_CATTLE_NO', 'MOTHER_ANIMAL_NO', 'F_GMOTHER_ANIMAL_NO', 'F_GFATHER_CATTLE_NO', 'M_GMOTHER_ANIMAL_NO', 'M_GFATHER_CATTLE_NO', 'C2023', 'C2024', 'C2025', 'AREA']

weather_df 컬럼: ['stn', 'date', 'ta_min', 'ta_max', 'rn_day', 'rhm_avg', 'ws_davg', 'THI', 'THI_grade']
BIRTH_YMD 결측: 0
BIRTH_YMD 범위: 1996-01-01 00:00:00 ~ 2025-08-26 00:00:00
shape: (2408699, 35)


In [2]:
# 날짜 변환
train_df["ABATT_DATE"] = pandas.to_datetime(train_df["ABATT_DATE"])
weather_df["date"]     = pandas.to_datetime(weather_df["date"])

In [3]:
import numpy as np

# THI 등급 더미 변수
for g in ["양호", "주의", "경고", "위험", "폐사"]:
    weather_df[f"d_{g}"] = (weather_df["THI_grade"] == g).astype(int)

sum_cols = ["d_양호", "d_주의", "d_경고", "d_위험", "d_폐사",
            "rn_day", "ws_davg", "ta_min"]

train_df = train_df.reset_index(drop=True)
n = len(train_df)

# 결과 배열 초기화
days_arr = np.full(n, np.nan)
seg_arr  = np.full((n, len(sum_cols)), np.nan)

# 관측소별 누적합 + searchsorted
print("집계 시작...")
for stn, w in weather_df.groupby("stn"):
    w = w.sort_values("date")
    dates = w["date"].values

    csum = np.vstack([
        np.zeros(len(sum_cols)),
        w[sum_cols].cumsum().values
    ])

    idx = np.where(train_df["stn"].values == stn)[0]
    if len(idx) == 0:
        continue

    births = train_df["BIRTH_YMD"].values[idx]
    abatts = train_df["ABATT_DATE"].values[idx]

    lo = np.searchsorted(dates, births, side="left")
    hi = np.searchsorted(dates, abatts, side="right")

    seg_arr[idx]  = csum[hi] - csum[lo]
    days_arr[idx] = (hi - lo)

print("집계 완료")

# 결과 DataFrame
weather_features = pandas.DataFrame({
    "CATTLE_NO":  train_df["CATTLE_NO"],
    "days_total": days_arr,
    "days_양호":  seg_arr[:, 0],
    "days_주의":  seg_arr[:, 1],
    "days_경고":  seg_arr[:, 2],
    "days_위험":  seg_arr[:, 3],
    "days_폐사":  seg_arr[:, 4],
})

with np.errstate(invalid="ignore", divide="ignore"):
    weather_features["rn_day_mean"]  = seg_arr[:, 5] / days_arr
    weather_features["ws_davg_mean"] = seg_arr[:, 6] / days_arr
    weather_features["ta_min_mean"]  = seg_arr[:, 7] / days_arr

print(f"완료: {weather_features.shape}")
print(weather_features.head())

# 검증
check = (weather_features[["days_양호","days_주의","days_경고","days_위험","days_폐사"]].sum(axis=1)
         == weather_features["days_total"])
print(f"\n검증 (등급합==총일수): {check.sum():,} / {len(weather_features):,}")
print(f"days_total 결측: {weather_features['days_total'].isnull().sum():,}")

집계 시작...
집계 완료
완료: (2408699, 10)
                                      CATTLE_NO  days_total  days_양호  days_주의  \
0  mI4i8G0BJ8kWD6gdm77RmTzyuIx6N2ZMaZ7wFXx3xb4=      1097.0    750.0    143.0   
1  tMgfi1p35taO9GH4XN4x0bfO8czy79B8V9NyczRuV+8=       973.0    645.0    151.0   
2  mdsSD4/sB06U68LNT6P/GUwei2ISyIBZ+R04iLw2mX0=       882.0    545.0    139.0   
3  LZWd4YatUD851LyC/LFXuxPW7KEXTwVWQONOwqrkLPI=       967.0    641.0    149.0   
4  A4FF/dhr8vg7JeW/eqc3922pA69nEaLL/+uoAusZqyU=       966.0    640.0    149.0   

   days_경고  days_위험  days_폐사  rn_day_mean  ws_davg_mean  ta_min_mean  
0    197.0      7.0      0.0     3.291705      2.200273     5.592343  
1    169.0      8.0      0.0     3.641829      1.932477     4.339774  
2    188.0     10.0      0.0     2.943311      0.706474     5.978231  
3    169.0      8.0      0.0     3.630817      1.931851     4.324716  
4    169.0      8.0      0.0     3.634058      1.930538     4.317702  

검증 (등급합==총일수): 2,408,699 / 2,408,699
days_total 결측: 0

In [4]:
# train_df에 병합
before_cols = len(train_df.columns)

train_df = train_df.merge(weather_features, on="CATTLE_NO", how="left")

print(f"병합 전: {before_cols}열")
print(f"병합 후: {len(train_df.columns)}열")
print(f"행 수: {len(train_df):,}")

병합 전: 35열
병합 후: 44열
행 수: 2,408,699


In [6]:
train_df.to_csv("../data/processed/step_4-4_weather.csv",
                index=False, encoding="utf-8-sig")
print(f"저장 완료: {train_df.shape}")

저장 완료: (2408699, 44)
